# SensIA — nove macroclasses: dados, treino e avaliação
Entrada: `content_metadata.description` → E5 congelado → MLP treinada → `content_parent_category`.

Decisão da equipe: aceitar os rótulos do FineVideo, sem EDA nem revisão semântica manual.
O notebook apenas verifica formato, vazios, duplicatas e separação dos conjuntos; não corrige rótulos.

RAG e integração com vídeos não são executados aqui.

In [ ]:
%pip install -q "huggingface_hub>=0.27,<2" "pyarrow>=18,<24" "sentence-transformers>=3,<6" "scikit-learn>=1.5,<2" pandas matplotlib joblib threadpoolctl

## 1. Configuração
Use o secret `HF_TOKEN` do Colab e aceite as condições do FineVideo. Uma GPU T4 é opcional para os embeddings; o classificador sklearn roda em CPU.
A pasta desta versão é diferente da experiência de três classes, preservando seus arquivos anteriores.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
from collections import Counter
import json, re, hashlib, unicodedata, copy, time, platform
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display
from huggingface_hub import HfApi, HfFileSystem
import pyarrow.parquet as pq

SEED = 42
META_POR_CLASSE = 500
MINIMO_POR_CLASSE = 200
MIN_PALAVRAS = 8
# Taxonomia original: não inferir macroclasse a partir da categoria fina.
ROTULOS_PT = {
    'Entertainment': 'Entretenimento',
    'Education': 'Educação',
    'Lifestyle': 'Estilo de vida',
    'News & Politics': 'Notícias e política',
    'Sports': 'Esportes',
    'Art & Creativity': 'Arte e criatividade',
    'Science & Technology': 'Ciência e tecnologia',
    'Automotive': 'Automotivo',
    'Hobbies & Interests': 'Hobbies e interesses',
}
CLASSES = sorted(ROTULOS_PT)
assert len(CLASSES) == 9

drive.mount('/content/drive')
PASTA = Path('/content/drive/MyDrive/SensIA/rede_macro_9classes')
PASTA.mkdir(parents=True, exist_ok=True)
TOKEN = userdata.get('HF_TOKEN')
REPO = 'HuggingFaceFV/finevideo'
api = HfApi(token=TOKEN)
info = api.dataset_info(REPO)
REVISAO = info.sha
fs = HfFileSystem(token=TOKEN)
# Ordem pseudoaleatória reproduzível de arquivos, sem usar rótulos/teste.
ARQUIVOS = sorted(f.rfilename for f in info.siblings if f.rfilename.endswith('.parquet'))
ARQUIVOS = list(np.random.default_rng(SEED).permutation(ARQUIVOS))
MAX_ARQUIVOS = len(ARQUIVOS)  # limite superior; coleta para ao atingir a meta válida
COLETA = PASTA / f'coleta_{REVISAO[:12]}'
COLETA.mkdir(exist_ok=True)
print('Revisão:', REVISAO)
print('Meta:', META_POR_CLASSE, 'por macroclasse; sem baixar a coluna mp4.')
display(pd.DataFrame([{'rotulo_original': c, 'nome_pt': ROTULOS_PT[c]} for c in CLASSES]))

Mounted at /content/drive
Revisão: 84c74091e1c6ee7a5dffabfafb5c9033e4718883
Meta: 500 por macroclasse; sem baixar a coluna mp4.


,rotulo_original,nome_pt
0,Art & Creativity,Arte e criatividade
1,Automotive,Automotivo
2,Education,Educação
3,Entertainment,Entretenimento
4,Hobbies & Interests,Hobbies e interesses
5,Lifestyle,Estilo de vida
6,News & Politics,Notícias e política
7,Science & Technology,Ciência e tecnologia
8,Sports,Esportes


## 2. Funções de preparação
Não removemos stopwords nem palavras que indiquem naturalmente o tema. O rótulo, título, canal e transcrição não são concatenados à descrição da entrada.
A classificação continua sendo de assunto, não de estímulo/bem-estar. Descrições e rótulos gerados automaticamente são aceitos como fornecidos, conforme a decisão do projeto.

In [ ]:
def limpar_texto(valor):
    if not isinstance(valor, str): return ''
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', valor)).strip()

def chave_texto(texto):
    return re.sub(r'[^\w]+', ' ', texto.casefold()).strip()

def preparar_base(registros):
    df = pd.DataFrame(registros)
    if df.empty: return df
    df['descricao_global'] = df['descricao_global'].map(limpar_texto)
    df['chave'] = df['descricao_global'].map(chave_texto)
    df = df[(df.descricao_global.str.split().str.len() >= MIN_PALAVRAS)
            & df.video_id.ne('') & df.categoria.isin(CLASSES)].copy()
    for coluna in ['video_id', 'chave']:
        conflitos = df.groupby(coluna).categoria.nunique()
        df = df[~df[coluna].isin(conflitos[conflitos > 1].index)].copy()
    return df.drop_duplicates('video_id').drop_duplicates('chave').reset_index(drop=True)

def extrair_registro(meta, origem):
    categoria = limpar_texto(meta.get('content_parent_category'))
    if categoria not in CLASSES: return None
    identificador = meta.get('original_json_filename') or meta.get('original_video_filename')
    descricao = limpar_texto((meta.get('content_metadata') or {}).get('description'))
    if not identificador or len(descricao.split()) < MIN_PALAVRAS: return None
    return {'video_id': Path(str(identificador)).stem,
            'descricao_global': descricao, 'categoria': categoria,
            'categoria_fina_original': limpar_texto(meta.get('content_fine_category')),
            'canal': limpar_texto(meta.get('youtube_channel')),
            'titulo': limpar_texto(meta.get('youtube_title')),
            'url_video': 'https://www.youtube.com/watch?v=' + Path(str(identificador)).stem,
            'arquivo_origem': origem, 'revisao_dataset': REVISAO,
            'origem_texto': 'content_metadata.description'}

## 3. Coleta retomável: somente JSON
A célula lê somente a coluna `json` dos Parquets remotos. Pode haver transferência de overhead dos arquivos. Cada arquivo concluído gera um checkpoint, inclusive se não contiver registros úteis; repetir a célula retoma o trabalho.
A coleta considera as nove classes e para quando cada uma tem a meta de registros estruturalmente válidos. Não reutiliza o cache antigo de três categorias. A leitura remota pode demorar; não há estimativa fixa de tempo.

In [ ]:
registros = []
lidos = set()
for cache in sorted(COLETA.glob('parte_*.json')):
    salvo = json.loads(cache.read_text(encoding='utf-8'))
    lidos.add(salvo['arquivo'])
    registros.extend(salvo['registros'])

for nome in ARQUIVOS[:MAX_ARQUIVOS]:
    base_atual = preparar_base(registros)
    contagens = (base_atual.categoria.value_counts() if not base_atual.empty else pd.Series(dtype=int))
    if all(contagens.get(c, 0) >= META_POR_CLASSE for c in CLASSES): break
    if nome in lidos: continue
    novos = []
    try:
        with fs.open(f'datasets/{REPO}@{REVISAO}/{nome}', 'rb',
                     block_size=256*1024, cache_type='readahead') as remoto:
            tabela = pq.ParquetFile(remoto, pre_buffer=False).read(columns=['json'], use_threads=False)
            for linha in tabela.to_pylist():
                valor = linha['json']
                if isinstance(valor, bytes): valor = valor.decode('utf-8')
                meta = valor if isinstance(valor, dict) else json.loads(valor)
                registro = extrair_registro(meta, nome)
                if registro is not None: novos.append(registro)
        cache = COLETA / ('parte_' + hashlib.sha256(nome.encode()).hexdigest()[:20] + '.json')
        temporario = cache.with_suffix('.tmp')
        temporario.write_text(json.dumps({'arquivo': nome, 'registros': novos}, ensure_ascii=False), encoding='utf-8')
        temporario.replace(cache)
        registros.extend(novos)
        lidos.add(nome)
        if len(lidos) % 25 == 0:
            print(len(lidos), 'arquivos concluídos; registros coletados:', len(registros), flush=True)
    except Exception as erro:
        raise RuntimeError(f'Coleta interrompida em {nome}; progresso anterior preservado. Reexecute após resolver acesso/conexão. {type(erro).__name__}: {erro}') from erro

base_total = preparar_base(registros)
if base_total.empty: raise RuntimeError('Sem dados. Confira HF_TOKEN, aceite e acesso.')
contagens = base_total.categoria.value_counts().reindex(CLASSES, fill_value=0)
# É uma verificação operacional de tamanho, não EDA ou revisão de rótulos.
print('Registros válidos disponíveis por classe:', contagens.to_dict())
if contagens.min() < MINIMO_POR_CLASSE:
    raise RuntimeError('Alguma macroclasse ficou abaixo do mínimo operacional. Amplie MAX_ARQUIVOS se limitado; se todo o dataset já foi lido, revise o tamanho mínimo com os mentores. Nenhuma classe será descartada silenciosamente.')
quantidade = min(META_POR_CLASSE, int(contagens.min()))
base = pd.concat([base_total[base_total.categoria == c].sample(n=quantidade, random_state=SEED)
                  for c in CLASSES], ignore_index=True)
base.to_csv(PASTA/'dataset_macro.csv', index=False, encoding='utf-8-sig')
manifesto = {'dataset': REPO, 'revisao': REVISAO, 'seed': SEED,
             'entrada': 'content_metadata.description', 'alvo': 'content_parent_category',
             'meta_por_classe': META_POR_CLASSE, 'selecionados_por_classe': quantidade,
             'total_selecionado': len(base), 'arquivos_lidos': len(lidos),
             'rotulos': ROTULOS_PT, 'eda': False, 'revisao_semantica': False,
             'decisao': 'Rótulos originais aceitos conforme orientação dos mentores',
             'proveniencia': 'FineVideo/YouTube Commons; consultar licenças originais e termos do dataset',
             'termos': 'https://huggingface.co/datasets/HuggingFaceFV/finevideo'}
(PASTA/'manifesto_dataset.json').write_text(json.dumps(manifesto, ensure_ascii=False, indent=2), encoding='utf-8')
print('Base preparada:', len(base), 'exemplos;', quantidade, 'por classe. Não há avaliação do modelo ainda.')

25 arquivos concluídos; registros coletados: 795
50 arquivos concluídos; registros coletados: 1597
75 arquivos concluídos; registros coletados: 2393
100 arquivos concluídos; registros coletados: 3190
125 arquivos concluídos; registros coletados: 3991
150 arquivos concluídos; registros coletados: 4790
175 arquivos concluídos; registros coletados: 5587
200 arquivos concluídos; registros coletados: 6385
225 arquivos concluídos; registros coletados: 7185
250 arquivos concluídos; registros coletados: 7986
275 arquivos concluídos; registros coletados: 8787
300 arquivos concluídos; registros coletados: 9588
325 arquivos concluídos; registros coletados: 10384
350 arquivos concluídos; registros coletados: 11182
375 arquivos concluídos; registros coletados: 11986
400 arquivos concluídos; registros coletados: 12782
425 arquivos concluídos; registros coletados: 13582
Registros válidos disponíveis por classe: {'Art & Creativity': 992, 'Automotive': 500, 'Education': 2083, 'Entertainment': 2967, 'Ho

## 4. Treino/validação/teste sem compartilhar canais
Aproximadamente 70/15/15. A escolha da divisão usa somente tamanhos e rótulos, nunca desempenho de modelos. Duplicatas exatas foram removidas; textos semanticamente parecidos não são auditados.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def separar_base(base):
    df = base.reset_index(drop=True).copy()
    df['grupo'] = df.apply(lambda r: 'canal:' + r['canal'].casefold() if r['canal'] else 'video:' + r['video_id'], axis=1)
    melhor = None
    for seed in range(SEED, SEED + 100):
        a, b = next(GroupShuffleSplit(n_splits=1, train_size=.7, random_state=seed).split(df, groups=df.grupo))
        resto = df.iloc[b]
        if resto.grupo.nunique() < 2: continue
        v, t = next(GroupShuffleSplit(n_splits=1, test_size=.5, random_state=seed).split(resto, groups=resto.grupo))
        partes = {'treino': df.iloc[a].copy(), 'validacao': resto.iloc[v].copy(), 'teste': resto.iloc[t].copy()}
        if any((p.categoria.value_counts().reindex(CLASSES, fill_value=0) < 5).any() for p in partes.values()): continue
        custo = sum(abs(len(partes[s])/len(df)-alvo) for s,alvo in [('treino',.7),('validacao',.15),('teste',.15)])
        custo += sum(float((p.categoria.value_counts(normalize=True).reindex(CLASSES,fill_value=0)-1/len(CLASSES)).abs().sum()) for p in partes.values())
        if melhor is None or custo < melhor[0]: melhor = (custo, partes)
    if melhor is None: raise ValueError('Não foi possível separar as nove classes por canais. Colete mais dados; não mude silenciosamente para divisão aleatória por linha.')
    saida = []
    for split, parte in melhor[1].items():
        parte['split'] = split
        saida.append(parte)
    dados = pd.concat(saida, ignore_index=True)
    for coluna in ['video_id', 'chave', 'grupo']:
        assert dados.groupby(coluna).split.nunique().max() == 1
    return dados

dados = separar_base(base)
for split in ['treino', 'validacao', 'teste']:
    dados[dados.split == split].to_csv(PASTA/f'{split}.csv', index=False, encoding='utf-8-sig')
display(pd.crosstab(dados.split, dados.categoria).reindex(columns=CLASSES))

categoria,Art & Creativity,Automotive,Education,Entertainment,Hobbies & Interests,Lifestyle,News & Politics,Science & Technology,Sports
split,,,,,,,,,
teste,76,78,74,82,76,71,84,71,72
treino,346,362,352,346,345,357,339,357,355
validacao,78,60,74,72,79,72,77,72,73
